# Cloud Press 텍스트 모델 — LoRA 파인튜닝 3종 (Cloudflare Workers AI용)

Workers AI의 LoRA 지원 기반 모델 중 3개를 파인튜닝해 Workers AI에 올립니다. 결과는 `inference-edge` Worker가 `profile`/`mode`에 따라 자동으로 사용합니다.

| 프로필 | 기반 모델 (HF) | Workers AI 모델 | 용도 |
|---|---|---|---|
| `quality` | `google/gemma-7b-it` | `@cf/google/gemma-7b-it-lora` | 기본·품질 (한국어 토크나이저가 3개 중 가장 낫다) |
| `general` | `mistralai/Mistral-7B-Instruct-v0.2` | `@cf/mistral/mistral-7b-instruct-v0.2-lora` | 범용·영어 |
| `fast` | `google/gemma-2b-it` | `@cf/google/gemma-2b-it-lora` | 가장 가벼워 같은 무료 한도로 가장 많이 생성 |

**알아둘 점**
- Workers AI에서 LoRA를 쓸 수 있는 기반 모델은 카탈로그의 "LoRA" 태그 모델뿐입니다. Qwen·GLM·DeepSeek·Gemma 4 같은 최신 대형 모델은 LoRA를 지원하지 않아, 위 3개가 "LoRA 가능" 조건 안에서 고른 최선입니다.
- 어댑터는 rank ≤ 8(최대 32), 300MB 미만, 파일명 `adapter_config.json`/`adapter_model.safetensors`, `model_type` 기록이 필요합니다(스크립트가 자동 처리·검증).
- Gemma 모델은 Hugging Face에서 **라이선스 동의(승인)** 가 필요합니다: https://huggingface.co/google/gemma-7b-it , https://huggingface.co/google/gemma-2b-it

**준비물**: ① 학습 데이터(JSONL, Drive의 `cloud-press/text-lora/train.jsonl`) ② HF 토큰(read) ③ Cloudflare API 토큰(`Workers AI: Edit`) + Account ID

## 1. GPU 확인
7B 모델은 4비트(QLoRA)로 T4(16GB)에서도 가능하지만, Gemma는 fp16에서 불안정할 수 있어 **L4/A100(bf16 지원)** 을 권장합니다.

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16 지원:", torch.cuda.is_bf16_supported())

## 2. 설치

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate safetensors huggingface_hub

## 3. 리포지토리 + Drive + 경로

In [ ]:
!git clone https://github.com/choichoi3227-crypto/cloud-press.git 2>/dev/null || (cd cloud-press && git pull -q)
%cd /content/cloud-press/ai-models/training/text-lora

from google.colab import drive
drive.mount('/content/drive')
import os
BASE = "/content/drive/MyDrive/cloud-press/text-lora"
DATA = f"{BASE}/train.jsonl"          # ← 학습 데이터를 여기에 올려 두세요
OUT_ROOT = f"{BASE}/out"
os.makedirs(OUT_ROOT, exist_ok=True)
print("데이터 존재:", os.path.exists(DATA))

## 4. Hugging Face 로그인 (Gemma 접근용)

In [ ]:
from huggingface_hub import login
login()

## 5. 데이터 확인
형식 오류로 건너뛰는 줄이 있는지, 샘플이 어떻게 보이는지 확인합니다. 지원 형식: `instruction/input/output`, `prompt/response`, `messages`.

In [ ]:
!python train_text_lora.py --check-data --data {DATA} --base google/gemma-2b-it

## 6. 스모크 테스트 (3스텝, 약 몇 분)
**내장 샘플 데이터로 파이프라인만 확인**합니다(학습 품질과 무관). 이 결과는 Cloudflare에 올리지 마세요.

In [ ]:
!python train_text_lora.py --base google/gemma-2b-it --data sample_sft.jsonl --out /content/smoke --max-steps 3 --grad-accum 2

## 7. 학습 — fast (Gemma 2B)

In [ ]:
!python train_text_lora.py --base google/gemma-2b-it --data {DATA} --out {OUT_ROOT}/gemma2b

## 8. 학습 — quality (Gemma 7B)

In [ ]:
!python train_text_lora.py --base google/gemma-7b-it --data {DATA} --out {OUT_ROOT}/gemma7b

## 9. 학습 — general (Mistral 7B)

In [ ]:
!python train_text_lora.py --base mistralai/Mistral-7B-Instruct-v0.2 --data {DATA} --out {OUT_ROOT}/mistral7b

## 10. 어댑터 검증 (Workers AI 요구사항)
세 개 모두 `✅ 검증 통과`가 나와야 합니다.

In [ ]:
!python train_text_lora.py --validate-only --base google/gemma-2b-it --out {OUT_ROOT}/gemma2b
!python train_text_lora.py --validate-only --base google/gemma-7b-it --out {OUT_ROOT}/gemma7b
!python train_text_lora.py --validate-only --base mistralai/Mistral-7B-Instruct-v0.2 --out {OUT_ROOT}/mistral7b

## 11. Cloudflare에 업로드
API 토큰에는 `Workers AI: Edit` 권한이 필요합니다. 업로드 후에는 fine-tune 파일을 수정할 수 없고, 바꾸려면 새 이름으로 다시 만들어야 합니다.
(Worker는 어댑터가 없으면 자동으로 기반 모델만 호출하므로, 업로드 전에도 서비스는 동작합니다.)

In [ ]:
import os, getpass
os.environ["CLOUDFLARE_API_TOKEN"] = getpass.getpass("Cloudflare API 토큰: ")
os.environ["CLOUDFLARE_ACCOUNT_ID"] = input("Account ID: ").strip()
!npx -y wrangler@4 ai finetune create @cf/google/gemma-2b-it-lora cp-gemma2b {OUT_ROOT}/gemma2b
!npx -y wrangler@4 ai finetune create @cf/google/gemma-7b-it-lora cp-gemma7b {OUT_ROOT}/gemma7b
!npx -y wrangler@4 ai finetune create @cf/mistral/mistral-7b-instruct-v0.2-lora cp-mistral7b {OUT_ROOT}/mistral7b
!npx -y wrangler@4 ai finetune list

## 12. 실제 Workers AI에서 어댑터 동작 테스트
파인튜닝 적용 응답을 확인합니다. 4비트(QLoRA)로 학습한 어댑터는 Cloudflare의 비양자화 기반 모델에서 품질이 달라질 수 있으므로, **응답이 기대와 맞는지 꼭 확인**하세요(이상하면 `--no-4bit`로 A100에서 다시 학습).

In [ ]:
import requests, json
from train_text_lora import user_turn
acct, tok = os.environ["CLOUDFLARE_ACCOUNT_ID"], os.environ["CLOUDFLARE_API_TOKEN"]
tests = [("@cf/google/gemma-2b-it-lora","cp-gemma2b"),("@cf/google/gemma-7b-it-lora","cp-gemma7b"),("@cf/mistral/mistral-7b-instruct-v0.2-lora","cp-mistral7b")]
for model, lora in tests:
    r = requests.post(f"https://api.cloudflare.com/client/v4/accounts/{acct}/ai/run/{model}",
        headers={"Authorization": f"Bearer {tok}"},
        json={"messages":[{"role":"user","content":user_turn("블로그 글 제목 3개를 추천해줘. 주제: 초보자를 위한 재택근무 책상 정리")}],
              "lora": lora, "max_tokens": 200})
    print("=====", lora, r.status_code); print(json.dumps(r.json(), ensure_ascii=False)[:600])

## 13. 다음 단계
`inference-edge` Worker를 배포하면 `profile`(quality/general/fast) 또는 `mode`(fast→fast, thinking→quality)에 따라 위 어댑터가 자동 적용됩니다. 확인:
`curl .../v1/text/generate -d '{"prompt":"...","profile":"general"}'` → 응답의 `lora`가 `cp-mistral7b`면 어댑터 적용, `null`이면 기반 모델 폴백입니다.